In [4]:
!pip install -q transformers==4.46.1 IndicTransToolkit sentencepiece
print("Install finished")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.4/548.4 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 10.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour

In [1]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))
print("Logged in to Hugging Face")

Logged in to Hugging Face


In [2]:
import time
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

try:
    from IndicTransToolkit.processor import IndicProcessor
except ImportError:
    from IndicTransToolkit import IndicProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "ai4bharat/indictrans2-en-indic-dist-200M"
it2_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
it2_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16,
).to(DEVICE)
ip = IndicProcessor(inference=True)

input_sentences = [
    "Can you hear my voice?",
    "I don't know where to go. Please help me.",
]

for tgt_lang, name in [("hin_Deva", "Hindi"), ("tam_Taml", "Tamil"), ("tel_Telu", "Telugu")]:
    start = time.time()
    batch = ip.preprocess_batch(input_sentences, src_lang="eng_Latn", tgt_lang=tgt_lang)
    inputs = it2_tokenizer(
        batch, truncation=True, padding="longest",
        return_tensors="pt", return_attention_mask=True,
    ).to(DEVICE)

    with torch.no_grad():
        generated = it2_model.generate(
            **inputs, use_cache=True, min_length=0, max_length=256, num_beams=5,
        )

    decoded = it2_tokenizer.batch_decode(
        generated, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )
    translations = ip.postprocess_batch(decoded, lang=tgt_lang)

    print(f"--- IndicTrans2 {name} ({time.time() - start:.2f}s) ---")
    for t in translations:
        print(t)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

dict.TGT.json: 0.00B [00:00, ?B/s]

...

modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

--- IndicTrans2 Hindi (1.69s) ---
क्या आप मेरी आवाज़ सुन सकते हैं?
मुझे नहीं पता कि कहाँ जाना है। कृपया मेरी मदद करें।
--- IndicTrans2 Tamil (0.47s) ---
என் குரல் கேட்கிறதா?
எங்கு செல்வது என்று தெரியவில்லை. தயவுசெய்து எனக்கு உதவுங்கள்.
--- IndicTrans2 Telugu (0.44s) ---
నా గొంతు మీకు వినిపిస్తోందా?
ఎక్కడికి వెళ్ళాలో నాకు తెలియదు. దయచేసి నాకు సహాయం చేయండి.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name, torch_dtype=torch.float16
).to("cuda")
print("Translation model loaded")

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Translation model loaded


In [ ]:
def translate_from_english(sentences, tgt_lang):
    inputs = tokenizer(
        sentences, padding=True, truncation=True, max_length=256, return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
            num_beams=5,
            max_length=256,
        )
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [ ]:
import time

sentences = [
    "Can you hear my voice?",
    "I don't know where to go. Please help me.",
]

languages = [("hin_Deva", "Hindi"), ("tam_Taml", "Tamil"), ("tel_Telu", "Telugu")]

for code, name in languages:
    start = time.time()
    results = translate_from_english(sentences, code)
    print(f"--- {name} ({time.time() - start:.2f}s) ---")
    for r in results:
        print(r)

--- Hindi (1.20s) ---
क्या आप मेरी आवाज सुन सकते हैं?
मुझे नहीं पता कि कहाँ जाना है, कृपया मेरी मदद करें।
--- Tamil (0.31s) ---
நீங்கள் என் குரல் கேட்க முடியும்?
நான் எங்கே செல்ல வேண்டும் என்று எனக்கு தெரியாது. தயவு செய்து எனக்கு உதவுங்கள்.
--- Telugu (0.29s) ---
మీరు నా వాయిస్ వినవచ్చు?
నేను ఎక్కడికి వెళ్ళాలో నాకు తెలియదు. దయచేసి నాకు సహాయం చేయండి.


In [2]:
import time
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "ai4bharat/indictrans2-en-indic-dist-200M"
it2_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
it2_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16,
).to(DEVICE)
ip = IndicProcessor(inference=True)

input_sentences = [
    "Can you hear my voice?",
    "I don't know where to go. Please help me.",
]

for tgt_lang, name in [("hin_Deva", "Hindi"), ("tam_Taml", "Tamil"), ("tel_Telu", "Telugu")]:
    start = time.time()
    batch = ip.preprocess_batch(input_sentences, src_lang="eng_Latn", tgt_lang=tgt_lang)
    inputs = it2_tokenizer(
        batch, truncation=True, padding="longest",
        return_tensors="pt", return_attention_mask=True,
    ).to(DEVICE)

    with torch.no_grad():
        generated = it2_model.generate(
            **inputs, use_cache=True, min_length=0, max_length=256, num_beams=5,
        )

    decoded = it2_tokenizer.batch_decode(
        generated, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )
    translations = ip.postprocess_batch(decoded, lang=tgt_lang)

    print(f"--- IndicTrans2 {name} ({time.time() - start:.2f}s) ---")
    for t in translations:
        print(t)

ModuleNotFoundError: No module named 'IndicTransToolkit'